# 12-1 Evaluation Metrics for Logistic Regression

**Course:** Models of Statistical Analysis — Universidad de los Andes  
**Professor:** Alejandra Tabares

---

## Introduction

Once a logistic regression model has been fitted, we need to assess how well it classifies new observations. Unlike linear regression, where we can rely on $R^2$ or RMSE, classification models require a different set of tools.

The starting point for evaluating any binary classifier is the **confusion matrix**, which tallies the model's predictions against the true class labels across four categories:

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

- **True Positive (TP):** the model correctly predicted the positive class.
- **True Negative (TN):** the model correctly predicted the negative class.
- **False Positive (FP):** the model predicted positive, but the true label is negative (Type I error).
- **False Negative (FN):** the model predicted negative, but the true label is positive (Type II error).

The default classification threshold in `sklearn` is **0.5**: if $\hat{p} \geq 0.5$, predict class 1; otherwise predict class 0.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

np.random.seed(42)
print("Libraries loaded successfully.")

## 1. Generating Synthetic Data and Fitting a Logistic Regression Model

We simulate a binary classification dataset with 1,000 observations and 5 informative features. The dataset is split into a training set (70 %) and a test set (30 %).

In [ ]:
# ── Simulate data ─────────────────────────────────────────────────────────────
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_classes=2,
    class_sep=0.8,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Class balance (test) — 0: {(y_test==0).sum()}, 1: {(y_test==1).sum()}")

In [ ]:
# ── Fit logistic regression (no regularization — C very large) ────────────────
model = LogisticRegression(C=1e6, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)           # hard class predictions
y_prob  = model.predict_proba(X_test)[:, 1]  # predicted probabilities for class 1

print("Model fitted.")
print(f"First 10 predicted probabilities: {np.round(y_prob[:10], 3)}")

## 2. Confusion Matrix

The confusion matrix provides a complete picture of how the classifier performs on each class. Let us compute it and visualize it as a heatmap.

In [ ]:
# ── Compute confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"\nTN={tn}  FP={fp}  FN={fn}  TP={tp}")

In [ ]:
# ── Visualize confusion matrix ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"],
    ax=ax,
    linewidths=0.5,
    linecolor="gray",
)

ax.set_title("Confusion Matrix — Logistic Regression", fontsize=13, pad=12)
ax.set_ylabel("True Label", fontsize=11)
ax.set_xlabel("Predicted Label", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Classification Metrics: Accuracy, Precision, Recall, F1-Score

From the four cells of the confusion matrix we can derive the following scalar metrics:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP}$$

$$\text{Recall (Sensitivity)} = \frac{TP}{TP + FN}$$

$$\text{Specificity} = \frac{TN}{TN + FP}$$

$$\text{F1-Score} = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Key trade-off:** Precision and Recall often move in opposite directions. In medical diagnosis we typically prioritize **Recall** (avoid missing sick patients), whereas in spam filtering we may prioritize **Precision** (avoid flagging legitimate emails). The F1-Score provides a single balanced summary when both errors carry similar costs.

> **Note:** Accuracy can be misleading when classes are imbalanced. For example, if 95 % of observations belong to class 0, a model that always predicts 0 achieves 95 % accuracy while being completely useless for detecting class 1.

In [ ]:
# ── Compute metrics manually from confusion matrix ────────────────────────────
accuracy    = (tp + tn) / (tp + tn + fp + fn)
precision   = tp / (tp + fp)
recall      = tp / (tp + fn)
specificity = tn / (tn + fp)
f1          = 2 * precision * recall / (precision + recall)

print("Manual computation from confusion matrix:")
print(f"  Accuracy    : {accuracy:.4f}")
print(f"  Precision   : {precision:.4f}")
print(f"  Recall      : {recall:.4f}")
print(f"  Specificity : {specificity:.4f}")
print(f"  F1-Score    : {f1:.4f}")

In [ ]:
# ── sklearn classification_report ────────────────────────────────────────────
print("Classification Report (sklearn):")
print(classification_report(y_test, y_pred, target_names=["Class 0", "Class 1"]))

The `classification_report` provides precision, recall, and F1-score for **each class** as well as macro and weighted averages.

- **Macro average**: unweighted mean of per-class metrics — treats all classes equally regardless of their frequency.
- **Weighted average**: mean of per-class metrics weighted by the number of true instances per class — accounts for class imbalance.

## 4. ROC Curve and AUC

The metrics above depend on the chosen classification threshold (default 0.5). The **Receiver Operating Characteristic (ROC) curve** is a threshold-independent diagnostic tool that plots:

- **True Positive Rate (Sensitivity / Recall)** on the y-axis: $TPR = \frac{TP}{TP + FN}$
- **False Positive Rate (1 − Specificity)** on the x-axis: $FPR = \frac{FP}{FP + TN}$

as the threshold varies from 1 down to 0.

The **Area Under the Curve (AUC)** summarises the ROC curve with a single number:
- AUC = 0.5 → random classifier (diagonal line)
- AUC = 1.0 → perfect classifier (upper-left corner)
- AUC > 0.7 is generally considered acceptable; > 0.8 is good; > 0.9 is excellent

Probabilistic interpretation: AUC is the probability that the model assigns a **higher score to a randomly chosen positive example than to a randomly chosen negative example**.

In [ ]:
# ── ROC curve for the main model ──────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))

ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"Logistic Regression (AUC = {auc_score:.3f})")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random Classifier (AUC = 0.500)")

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate (1 − Specificity)", fontsize=11)
ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
ax.set_title("ROC Curve — Logistic Regression", fontsize=13)
ax.legend(loc="lower right", fontsize=10)
ax.fill_between(fpr, tpr, alpha=0.08, color="steelblue")

plt.tight_layout()
plt.show()

print(f"AUC = {auc_score:.4f}")

## 5. Comparing Two Models on the ROC Plot

A common use of the ROC curve is to compare competing models. We now fit a second model **with strong L2 regularization** (small `C`) and overlay both ROC curves.

In [ ]:
# ── Model 1: no regularization (C=1e6) — already fitted as `model` ─────────
# ── Model 2: strong L2 regularization (C=0.01) ───────────────────────────────
model_reg = LogisticRegression(C=0.01, penalty="l2", max_iter=1000, random_state=42)
model_reg.fit(X_train, y_train)
y_prob_reg = model_reg.predict_proba(X_test)[:, 1]

fpr2, tpr2, _ = roc_curve(y_test, y_prob_reg)
auc2 = roc_auc_score(y_test, y_prob_reg)

# ── Plot both ROC curves ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))

ax.plot(fpr,  tpr,  color="steelblue",   lw=2, label=f"No Regularization  (AUC = {auc_score:.3f})")
ax.plot(fpr2, tpr2, color="darkorange",  lw=2, label=f"L2 Reg. (C=0.01)   (AUC = {auc2:.3f})")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random Classifier")

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate (1 − Specificity)", fontsize=11)
ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
ax.set_title("ROC Curve Comparison: Regularization Effect", fontsize=13)
ax.legend(loc="lower right", fontsize=10)

plt.tight_layout()
plt.show()

print(f"Model 1 (no reg.)  AUC = {auc_score:.4f}")
print(f"Model 2 (L2 reg.)  AUC = {auc2:.4f}")
print(f"Difference in AUC  : {auc_score - auc2:+.4f}")

## 6. Summary Table

The cell below collects all computed metrics into a single summary table for easy comparison.

In [ ]:
# ── Compute metrics for both models ──────────────────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred_reg = model_reg.predict(X_test)

def metric_row(name, y_true, y_pred_labels, y_pred_proba):
    return {
        "Model"      : name,
        "Accuracy"   : accuracy_score(y_true, y_pred_labels),
        "Precision"  : precision_score(y_true, y_pred_labels),
        "Recall"     : recall_score(y_true, y_pred_labels),
        "F1-Score"   : f1_score(y_true, y_pred_labels),
        "AUC"        : roc_auc_score(y_true, y_pred_proba),
    }

summary = pd.DataFrame([
    metric_row("Logistic (no reg.)", y_test, y_pred,     y_prob),
    metric_row("Logistic (L2 reg.)", y_test, y_pred_reg, y_prob_reg),
])

summary = summary.set_index("Model")
print(summary.round(4).to_string())

## Key Takeaways

1. The **confusion matrix** is the foundation of all classification metrics. Always inspect it first.
2. **Accuracy** is intuitive but unreliable when classes are imbalanced.
3. **Precision** and **Recall** capture different types of errors; the right balance depends on the cost of each error in the application domain.
4. The **F1-Score** combines precision and recall into a single metric via the harmonic mean.
5. The **ROC curve** and **AUC** evaluate a model's discriminative ability across all possible thresholds, making them more robust for model comparison.
6. A higher AUC does not always translate into better predictions at the operational threshold — always examine the full confusion matrix at the threshold you plan to deploy.